# Phase 1: LLM Foundations

## Section 1: RLHF (Reinforcement Learning from Human Feedback)

### Overview
RLHF is the three-stage pipeline that transformed LLMs from next-token predictors into instruction-following agents. Think of it as wrapping supervised learning in a feedback loop. The core problem is LLMs predict likely next tokens, but likely doesnt mean ideal. RLHF uses human preference data to bridge the gap. 

### Stage 1: Supervised Fine-Tuning (SFT)
Start with a pretrained LLM (e.g., GPT-2). Fine-tune on high-quality, small dataset of demonstrations showing human written examples of dieal responses to prompts. This gets the model into roughly the right behavioral regime. 

- **Input:** "What's the capital of France?"
- **Target:** "The capital of France is Paris."

This is standard supervised learning—cross-entropy loss, backprop. The model learns to follow instructions.

**Why it works:** The base model has learned language structure; SFT teaches it *which* structures to produce in response to instructions.

### Stage 2: Reward Model Training
You now have an SFT model, but it doesn't know what humans actually prefer. Enter the reward model. Here we take the SFT model and generate multiple responses to the same prompt. Humans rank these responses. Then a seperate reeward mdoel is trained to predict these rankings. It takes a (prompt, response) pair and outputs a scalar score representing how good a response is.

Create a new dataset where humans rank pairs of responses (A vs B, or rate on a scale). Train a classifier:
- **Input:** (prompt, response)
- **Output:** scalar score (higher = better)

Think of this like training a regression model where the target is "human preference." The architecture reuses the SFT model's backbone but adds a scoring head.

**Key insight:** This reward model becomes your optimization objective. You're not optimizing cross-entropy anymore—you're optimizing for human-preferred outputs.

### Stage 3: PPO-Based Reinforcement Learning
Now use the reward model as a signal to fine-tune the SFT model via reinforcement learning. Here we use the reward model as a fitness function. The SFT modeel generates responses, the reward model scores them, and PPO (Proximal Policy Optimization) updtes the weights to prouce higher scoring outputs. A KL divergence penalty keeps the model from drifting too far from the SFT model (otherwise it can find weird ways to game the reward model).

**Algorithm: Proximal Policy Optimization (PPO)**
- **Policy:** Your SFT model (generates tokens)
- **Reward:** Your reward model (scores outputs)
- **Goal:** Maximize expected reward while staying close to the original SFT model (via KL divergence penalty)

The update rule, simplified:
```
loss = -E[log(π(a|s)) * r(s, a)] + β * KL(π_new || π_sft)
```

Where:
- `π(a|s)` = probability of action (token) a given state (context) s
- `r(s, a)` = reward model's score for that output
- `KL` = penalty to avoid diverging too far from SFT (prevents reward hacking)

**Why PPO?** It's stable and sample-efficient—you don't need millions of RL samples, and it converges without exploding gradients.

### Why its unstable in practice
RL is notoriously finicky - sensitive to hyperparameters, prone to reward hacking (model learns to exploit reward model quirks ratehr than improve), and computationally expensive. 

### The Three-Stage Pipeline (ASCII Diagram)

```
┌─────────────────────────────────────────────────────────────────┐
│ PRETRAINED LLM (e.g., GPT-2, LLaMA, Mistral)                    │
└──────────────────────┬──────────────────────────────────────────┘
                       │
                       ▼
         ┌─────────────────────────────┐
         │ STAGE 1: SFT                │
         │ Input: (prompt, response)   │
         │ Loss: Cross-entropy         │
         │ Output: SFT Model           │
         └──────────────┬──────────────┘
                        │
          ┌─────────────┴─────────────┐
          │                           │
          ▼                           ▼
┌──────────────────────┐   ┌─────────────────────────────┐
│ STAGE 2:             │   │                             │
│ Reward Model         │   │ STAGE 3: PPO-RL             │
│ Input: (prompt,resp) │   │ Policy: SFT Model           │
│ Output: score        │   │ Reward: Reward Model        │
│                      │   │ Optimization: Maximize      │
│ (Trained on human    │   │  expected reward - KL       │
│  preference labels)  │   │                             │
│                      │   │ Output: RLHF-tuned LLM      │
└──────────────────────┘   └─────────────────────────────┘
```

### Why This Matters

**Pre-RLHF:** Language models were next-token predictors. They'd complete "Explain quantum computing" with *any plausible continuation*—often incoherent or unhelpful.

**Post-RLHF:** Models align with human intentions. GPT-3.5 with RLHF beats GPT-3 without it on nearly every benchmark, despite similar or smaller scale.

**For AI Engineer interviews:** You should be able to explain:
1. Why SFT alone isn't enough (model doesn't know what humans prefer)
2. Why you need a reward model (human feedback is expensive; amortize it across the model)
3. Why PPO instead of simpler RL (stability, sample efficiency, KL regularization)
4. Trade-offs: reward hacking (gaming the reward model), "boring" output collapse, KL penalty tuning

### Connection to Your Background
- **SFT** = supervised learning you already know
- **Reward model** = training a classifier on preference data (binary or continuous targets)
- **PPO** = policy gradient RL with a trust region or just "optimize for a reward signal with a regularizer to stay close to the original model"


## Section 2: DPO (Direct Preference Optimization)

### The Problem with RLHF
RLHF is powerful but complex:
1. **Three separate models** (pretrained, SFT, reward) = memory and training overhead
2. **Reward model quality** directly impacts final model, trained on limited human feedback
3. **Instability:** PPO hard to tune; reward hacking is a real problem
4. **Computational cost:** Three sequential training passes

**What if we could optimize directly for preferences without a reward model?**

### The DPO Insight
DPO was designed to fix RLHFs instability. The key insight is the entire RLHF objective has a mathematical closed form solution that lets you skip the reward model and PPO step entirely.

Instead of training a reward model, you directly use the preference pairs (chosen response vs rejected response) to update the l anguage model via a modified loss function. The math reformulates the RL objective into something that looks like classification loss. The model just needs to increase the relative probability gap between chosen and rejected responses. 

Skip the reward model. Optimize directly using preference data.

**Key realization:** Derive preference loss from Bradley-Terry model.

### The Math

Start with preference pairs:
- **Prompt:** "What is 2+2?"
- **Preferred:** "The answer is 4."
- **Dispreferred:** "The answer is 5."

Bradley-Terry model gives probability preferred is better:
```
P(y_w > y_l) = exp(r(x, y_w)) / (exp(r(x, y_w)) + exp(r(x, y_l)))
```

**DPO reparameterization:** Don't learn r separately. Express via policy:
```
r(x, y) = β * log(π(y|x) / π_ref(y|x))
```

**DPO loss (simplified):**
```
L = -E[log(σ(β * (log(π(y_w|x)/π_ref(y_w|x)) - log(π(y_l|x)/π_ref(y_l|x)))))]
```

**Intuition:**
- Higher prob on preferred → loss low
- Higher prob on dispreferred → loss high
- Log-ratio prevents diverging from reference (implicit regularization)

### Practical Implications
- One training stage instead of three
- No seperate reward model to maintain
- Far more stable
- DPO has become default for most open source fine tunes

### Why DPO Works
1. No reward model needed
2. Single training loop
3. Implicit regularization prevents reward hacking
4. Faster (one model vs three)
5. Empirically beats RLHF with less compute

### RLHF vs DPO

| Factor | RLHF | DPO |
|--------|------|-----|
| Models | 3 | 1 |
| Complexity | High | Low |
| Stability | PPO finicky | Direct loss |
| Pref data | Ranking | Binary pairs |
| 2024+ adoption | Established | Becoming standard |

### For Interviews
- Derive DPO from Bradley-Terry
- When to use: simpler, faster than RLHF
- Trade-off: DPO needs good preference data
- Mention variants: IPO, KTO


## Section 3: Constitutional AI / RLAIF (Reinforcement Learning from AI Feedback)

### The Human Feedback Bottleneck
RLHF and DPO require human feedback: expensive, slow, limited scale.

**What if AI generated the feedback instead?**

### The Critique-Revise Loop

```
Prompt → Initial Response → AI Critique → Revised Response → AI Judge → Preference Label
```

1. Generate response from SFT model
2. AI critic identifies issues
3. Same model generates revised response
4. AI judge: which is better? → Preference
5. Train on AI-generated preference using DPO

### The Constitution
Guide AI critic with principles:
- "Responses should be honest and truthful"
- "Avoid promoting harmful content"
- "Be helpful and direct"
- "If uncertain, say so"

Ensures consistent, scalable, customizable feedback.

### Why RLAIF Scales Better

| Aspect | RLHF | RLAIF |
|--------|------|-------|
| Speed | Slow | Fast (API calls) |
| Cost | High (~$1-5/label) | Low (~$0.01/label) |
| Consistency | Variable | Consistent |
| Scalability | ~10k labels | 1M+ labels |
| Customization | Hard | Easy (change constitution) |

### RLAIF vs RLHF: When to Use Each
- **RLAIF:** Scale to millions, have clear principles, objective tasks
- **RLHF:** Need ground truth, subjective tasks, cultural nuance
- **Hybrid:** Bootstrap with RLAIF (100k), then targeted RLHF (5-10k human)

### Claude's Training
Anthropic used Constitutional AI:
1. SFT on instruction data
2. Critique-revise with principles
3. DPO on AI preferences
4. Targeted RLHF

Result: Harmless, helpful, scalable.


## Section 4: Chinchilla Scaling Laws

### What is Chinchilla?
Chinchilla is a language model scaling study from DeepMind (2022) that revisited how model size and training data should scale together. Before Chinchilla, the common assumption (influenced by GPT-3) was that larger models were usually better, even if trained on relatively limited data.
The key finding was that many large language models were undertrained: they had too many parameters relative to the number of tokens seen during training.


### Compute-Optimal Training
Training cost can be approximated as:

[
\text{Compute} \propto N \times D
]

where:
(N) = number of model parameters
(D) = number of training tokens

Given a fixed compute budget, there is an optimal tradeoff between:
- Making the model larger
- Training on more data
- Chinchilla showed that previous large models allocated too much compute to parameter count and not enough to training tokens.

### The Famous “20 Tokens per Parameter” Rule
A rough practical takeaway from the paper:
For compute-optimal training, train on approximately 20 tokens per parameter.

Examples:


| Model Size | Compute-Optimal Tokens |
|--------|------|
| 1B params | ~20B tokens |
| 10B params | ~200B tokens |
| 70B params | ~1.4T tokens |



This is only a heuristic, not a fundamental law, but it became one of the most cited results from the paper.

### Why Was This Important?
#### Before Chinchilla:
GPT-3: 175B parameters
Trained on roughly 300B tokens
Only about 1.7 tokens per parameter

According to Chinchilla’s analysis, the compute would have been used more effectively by:
- Using a smaller model
- Training it on much more data
- DeepMind demonstrated that a 70B parameter model trained on ~1.4 trillion tokens could outperform much larger models trained on fewer tokens while using a similar compute budget.

#### The lesson:
More parameters are not always the best use of compute. A properly trained smaller model can outperform a larger undertrained model.


### Impact on Post-2022 LLM Training
Chinchilla significantly changed how frontier labs think about scaling:
**Pre-Chinchilla mindset**
- Increase parameter count aggressively
- Data scaling received less attention
- Models were often undertrained

**Post-Chinchilla mindset**
- Balance model size and dataset size
- Invest heavily in collecting and filtering trillions of tokens
- Train models longer rather than only making them bigger
- Optimize for compute efficiency rather than parameter count alone

### Interview Takeaway
If asked about Chinchilla in an AI engineer interview:
- Chinchilla introduced compute-optimal scaling laws.
- It showed many previous LLMs were undertrained relative to their size.
- The key heuristic is roughly 20 training tokens per parameter.
- For a fixed compute budget, a smaller model trained on more data can outperform a larger model trained on less data.
- The paper shifted industry focus from “just make models bigger” to “balance parameters and training data.”

### Note
Chinchilla scaling laws are strictly about pretraining.

The core finding was about the optimal allocation of a compute budget between model size and training tokens — specifically that previous models like GPT-3 were undertrained relative to their size. Chinchilla showed you should scale tokens and parameters roughly equally given a fixed compute budget.

This has nothing to do with:

- Fine-tuning (you're adapting an already pretrained model)
- RAG (you're retrieving at inference time, no training involved)
- Prompt engineering (no training at all)
- Inference (serving an already trained model)


## Section 5: Mixture of Experts (MoE)


A Mixture of Experts (MoE) model replaces a single feed-forward network with multiple **expert** networks.
A **router** decides which expert(s) should process each token.

Key idea:
> Not all parameters are used for every token.

This is called **conditional computation**.

Benefits:
- Much larger total parameter count
- Similar compute cost to a smaller dense model
- Better scaling efficiency

### Dense vs Sparse Routing


#### Dense Transformer

Every token uses the same feed-forward network.

```text
Token → FFN
```

All parameters are active for every token.

Characteristics:
- Simple implementation
- Predictable computation
- Compute grows directly with model size

#### MoE (Sparse Routing)

In an MoE layer, a router chooses only a few experts for each token.

```text
Token → Router → Top-k Experts
```

Most experts are skipped for any given token. The term sparse routing refers to the fact that only a sparse subset of experts is activated.

Characteristics:
- Only a fraction of parameters are active
- Significantly larger total model capacity - Capacity here means the total knowledge the model can store. More parameters = more capacity to memorise facts, patterns, and skills. A MoE model with 140B total parameters has roughly twice the storage capacity of a 70B dense model 
- Similar FLOPs to a smaller dense model - FLOPs (floating point operations) measure how much compute a forward pass requires — this determines speed and cost. If only 2 out of 16 experts are active per token, you're only doing compute through those 2 experts.

### Routing Mechanism
The router is typically a small neural network that produces a score for each expert.

Example:
- Expert 1: 0.10
- Expert 2: 0.55
- Expert 3: 0.05
- Expert 4: 0.30

For Top-2 routing, Selected experts:
- Expert 2
- Expert 4

The token is sent only to those experts. The outputs are then combined using the router weights.

Why not send every token to every expert?
Because that would eliminate the computational savings and effectively become a dense model.

### Memory vs Compute Tradeoff

A dense model with 100B parameters uses all 100B parameters every forward pass resulting in large memory footprint and compute.

An MoE model might have:

* 100B total parameters
* 10B active parameters per token

Result:

* **Memory:** similar to a 100B model
* **Compute:** closer to a 10B model

This results in large parameter count without proportional compute cost.



### Interview Takeaway

**Why use MoE?**

* Larger model capacity
* Lower compute per token
* Better scaling efficiency

**Main drawback:**

* High memory requirements
* Additional routing and communication complexity

**One-liner:**

> Mixture of Experts uses sparse routing to activate only a small subset of model parameters per token, trading memory for computational efficiency.


## Section 6: Modern Model Families



| Model Family | Open Weights? | Architecture | Training Approach | Key Strengths | Common Use Cases |
|-------------|--------------|--------------|-------------------|--------------|------------------|
| GPT-4o | No | Dense Transformer | Large-scale pretraining + instruction tuning + RLHF | Strong reasoning, coding, multimodal (text, image, audio) | General AI assistants, coding, enterprise applications |
| Claude | No | Dense Transformer | Large-scale pretraining + Constitutional AI + RLHF | Long context windows, reasoning, writing quality, safety | Analysis, document processing, writing |
| Llama 3 | Yes | Dense Transformer | Large-scale pretraining + instruction tuning | Strong open-weight performance, large ecosystem, fine-tuning flexibility | Research, self-hosting, custom AI systems |
| Mistral | Yes | Dense Transformer | Efficient large-scale pretraining + instruction tuning | Excellent performance-per-parameter, efficient inference | Resource-constrained deployments |
| Mixtral | Yes | Mixture of Experts (MoE) | MoE pretraining + sparse routing | Large effective capacity with lower compute cost | High-performance open-weight deployments |

### Interview takeaway:
- GPT-4o and Claude are leading closed-weight models.
- Llama 3, Mistral, and Mixtral are open-weight models.
- Mixtral differs architecturally by using Mixture of Experts (MoE) rather than a fully dense transformer.
- Open-weight models offer more customization; closed-weight models often provide stronger managed capabilities and tooling.

## Section 7: Context Windows

The context window is the total number of tokens the model can hold in memory at once — prompt tokens and generated tokens combined.

So if your context window is 200,000 tokens (Claude's current limit) and your prompt is 190,000 tokens, you only have 10,000 tokens left for generation before you hit the limit.
This has a direct practical implication for LLM Application Engineers — it's why context window management is a real engineering concern. In a multi-turn conversation the history keeps growing with every exchange. Eventually you hit the limit and have to decide what to drop. Common strategies are:

`Truncation` — drop the oldest messages first

`Summarisation` — compress old turns into a summary and replace them

`Sliding window` — always keep the system prompt and the last N turns

This also connects back to the KV cache — the cache grows with every generated token too, not just prompt tokens. Every token in the context window, regardless of whether it came from the prompt or was generated, has K and V vectors that need to be stored and attended to.

Examples:

| Era | Typical Context Window |
|------|----------------------|
| Early LLMs | 2k–4k tokens |
| Modern LLMs | 32k–128k tokens |
| Latest Models | 200k+ tokens |

Larger context windows allow models to work with longer documents, conversations, and retrieved knowledge.

### How Context Windows Grew

Increasing context length is challenging because self-attention scales approximately as:

O(n²)

where *n* is the number of tokens.

Key innovations that enabled longer context windows include:

- Improved positional encodings (e.g., RoPE)
- Attention optimizations and memory-efficient kernels
- Better training on long-context data
- Context-length extension during fine-tuning

These advances allowed context windows to grow from a few thousand tokens to hundreds of thousands of tokens.

### Lost in the Middle

Even when information fits inside the context window, models do not use all positions equally well.

**Lost in the Middle** refers to the tendency for models to:

- Pay more attention to information near the beginning
- Pay more attention to information near the end
- Miss information buried in the middle

As a result, retrieval quality can degrade even when the correct information is present in the prompt.

### Why It Matters for RAG

In Retrieval-Augmented Generation (RAG), retrieved documents are inserted into the model's context.

A common misconception is:

> Larger context windows automatically solve retrieval problems.

In practice:

- Important chunks may be buried in the middle of a long prompt
- Retrieval ranking becomes critical
- Chunk ordering matters
- More context is not always better context

A 200k-token context window is useful, but high-quality retrieval and prompt construction remain essential.

### Interview Takeaway

- Context windows have grown from ~4k tokens to 200k+ through improvements in attention mechanisms, positional encodings, and long-context training.
- Larger context windows enable processing longer documents and conversations.
- Models often exhibit a **lost in the middle** effect, where information in the middle of the context receives less attention.
- For RAG systems, retrieval quality and document ordering remain important even with very large context windows.

## Section 10: Fine-Tune vs RAG vs Prompting — When to Use Each

In practice, most LLM systems are built using one (or a combination) of:

- Prompt Engineering
- Retrieval-Augmented Generation (RAG)
- Fine-Tuning

The key interview skill is knowing **when each approach is appropriate**, not just how they work.

### Decision Framework

#### 1. Prompt Engineering

Use when:
- The model already has the knowledge
- You just need better structure, tone, or reasoning guidance

Why:
- Fastest and cheapest approach
- No training or infrastructure needed

Best for:
- Ad-hoc tasks
- Prototyping
- Simple behaviour control (formatting, tone, instructions)

---

#### 2. RAG (Retrieval-Augmented Generation)

Use when:
- The model does NOT have the required knowledge
- Data is private, proprietary, or frequently changing

Why:
- You inject knowledge at inference time
- No retraining required

Best for:
- Company documents
- Policies, manuals, knowledge bases
- Up-to-date information systems

#### 3. Fine-Tuning

Use when:
- You need consistent behaviour that prompting cannot reliably enforce
- You have labeled examples of desired input → output behaviour

Why:
- Adjusts model weights to learn patterns
- Improves consistency and style adherence

Best for:
- Structured outputs (classification, extraction formats)
- Domain-specific tone or writing style
- Repeated tasks at scale

### Comparison Table

| Approach | Cost | Latency | Data Required | When to Use |
|----------|------|----------|---------------|--------------|
| Prompt Engineering | Very low | Low | None | Model already knows task; need quick control or formatting |
| RAG | Medium (infra cost) | Higher (retrieval step) | Documents / knowledge base (unlabeled) | External or up-to-date knowledge not in model |
| Fine-Tuning | High (training cost) | Low at inference | Labeled input-output pairs | Need consistent behavior or style the model doesn't reliably follow via prompting |

### Why Interviewers Ask This

This question shows up constantly in AI Engineer interviews because it tests:

#### 1. System Design Thinking
- Can you choose the right tool for a production system?

#### 2. Practical Understanding
- Do you understand that "bigger model" is not always the solution?

#### 3. Tradeoff Awareness
- Cost vs latency vs performance vs maintainability

#### 4. Real-World Judgment
- Most real LLM systems are hybrids:
  - RAG + prompting is extremely common
  - Fine-tuning is used sparingly and deliberately

---

### Common Mistake Candidates Make

> “We should fine-tune for everything.”

In reality:
- Fine-tuning is expensive and harder to iterate on
- RAG is often the default for knowledge problems
- Prompting solves more than people expect

### Interview One-Liner

> Prompting is used for quick control, RAG is used for external knowledge, and fine-tuning is used for durable behavioural changes when prompting is not sufficient.